# Model Building on the Other stem first

In [17]:
import os
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio

In [18]:
##loading the other data(mel's) and lables
other_data = np.load(f'processed_data/other_features.npy')
other_labels = np.load(f'processed_data/other_labels.npy')

In [19]:
print(other_data.shape)
print(other_labels.shape)

(10198, 150, 150, 1)
(10198,)


## 3 way data split
80% Training: To teach the model.

10% Validation: To check for "Overfitting" during the training process.

10% Test: To give a final, unbiased accuracy score at the very end.

In [20]:
# shuffling is need cause we added the data in a sequential order first bollypop then carnatic so data inside data is in order
#stratify keep the proportion of data same in both test and train
from sklearn.model_selection import train_test_split

# this single line splits and shuffles at the same time
X_train, X_temp, Y_train, Y_temp = train_test_split(
    other_data,
    other_labels,
    test_size=0.2,
    random_state=42,
    shuffle=True, # be default it is true
    stratify=other_labels # Ensures 20% of EACH genre goes to the test set
)


#20 % remaining data in temp
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True, # be default it is true
    stratify= Y_temp # Ensures 20% of EACH genre goes to the test set
)
# Summary of the "Shuffling" Timeline
# During Split: Handled by train_test_split(shuffle=True). (Do this now)
# During Training: Handled by model.fit(shuffle=True). (Do this later)
# now ready for normalization

In [23]:
print(X_train.shape)
print(Y_train.shape)
print(X_val.shape)
print(Y_val.shape)
print(X_test.shape)
print(Y_test.shape)

(8158, 150, 150, 1)
(8158,)
(1020, 150, 150, 1)
(1020,)
(1020, 150, 150, 1)
(1020,)


## data normalization

In [24]:
other_data[:1]

# the values are in decibels(power_to_db) when converted to mel scale
# need to normalize the value form 0 to 1

array([[[[-48.452854],
         [-48.11557 ],
         [-49.23192 ],
         ...,
         [-47.95298 ],
         [-44.208584],
         [-39.58476 ]],

        [[-48.672886],
         [-43.46166 ],
         [-42.037247],
         ...,
         [-47.11456 ],
         [-45.67823 ],
         [-43.559166]],

        [[-54.899876],
         [-43.74206 ],
         [-40.584118],
         ...,
         [-45.917923],
         [-43.058113],
         [-40.126766]],

        ...,

        [[-79.950966],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.35068 ],
         [-63.332966]],

        [[-80.      ],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.50041 ],
         [-63.504673]],

        [[-80.      ],
         [-80.      ],
         [-80.      ],
         ...,
         [-80.      ],
         [-74.47734 ],
         [-63.471592]]]], dtype=float32)

## Normalization from (-80 - 0)  to  0 - 1 
The "Golden Rule": Split First, Normalize SecondMany beginners make the mistake of normalizing the entire dataset (all 10,000 samples) and then splitting it. This causes Data Leakage.Why? Normalization (like Min-Max Scaling) uses the min and max values of your data. If you normalize the whole set before splitting, your training data "knows" the maximum value of the test set. This "future information" leaks into your model, making your training accuracy look artificially high while your real-world performance fails.The Correct Way: 1.  Split your data into $X_{train}$ and $X_{test}$.2.  Calculate the min and max using only $X_{train}$.3.  Apply that same scaling to both $X_{train}$ and $X_{test}$.

In [25]:
# 1. Calculate stats from TRAINING data only
# This represents the "known world" to the model
X_min = X_train.min()
X_max = X_train.max()

# 2. Apply the same scaling to all three sets
# This squishes all dB values into a 0.0 to 1.0 range
X_train_norm = (X_train - X_min) / (X_max - X_min)
X_val_norm = (X_val - X_min) / (X_max - X_min)
X_test_norm = (X_test - X_min) / (X_max - X_min)

# 3. Verification - Essential Check!
print("--- Normalization Audit ---")
print(f"Train Range: {X_train_norm.min()} to {X_train_norm.max()}") # Must be 0.0 to 1.0
print(f"Val Range:   {X_val_norm.min():.2f} to {X_val_norm.max():.2f}")
print(f"Test Range:  {X_test_norm.min():.2f} to {X_test_norm.max():.2f}")

--- Normalization Audit ---
Train Range: 0.0 to 1.0
Val Range:   0.00 to 1.00
Test Range:  0.00 to 1.00


## Building the CNN Architecture